# Step 04: Interactive Hyperparameter Tuning & Sensitivity Analysis

Provides an easy-to-adjust control panel at the top to experiment with hyperparameters (`REPRESENTATION`, `lr`, `epochs`, `batch_size`) and instantly evaluate model performance.


In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
root_dir = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
if str(root_dir) not in sys.path: sys.path.insert(0, str(root_dir))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.train_loso import run_loso_cross_validation
sns.set_theme(style="whitegrid")

# ==============================================================================
# EASY HYPERPARAMETER ADJUSTMENT PANEL
# ==============================================================================
REPRESENTATION = "rep1_doppler"  # Options: 'rep1_doppler', 'rep2_projections', 'rep3_pointset'
epochs = 20                      # Number of training epochs per fold
lr = 5e-4                        # Learning rate (Adam optimizer)
batch_size = 32                  # Mini-batch size

print(f"Selected Representation: '{REPRESENTATION}'")
print(f"Selected Hyperparameters: epochs={epochs}, lr={lr}, batch_size={batch_size}")


In [ ]:
# Execute Experiment with Custom Hyperparameters
res = run_loso_cross_validation(rep_key=REPRESENTATION, epochs=epochs, lr=lr, batch_size=batch_size)
val_m = res["val_metrics"]

print(f"\n=== Validation Set Experiment Results ('{REPRESENTATION}') ===")
print(f"Validation Accuracy:   {val_m['accuracy']*100:.2f}%")
print(f"Validation Recall:     {val_m['recall']*100:.2f}%")
print(f"Validation Precision:  {val_m['precision']*100:.2f}%")
print(f"Validation Macro F1:   {val_m['macro_f1']*100:.2f}%")
print(f"Validation ROC-AUC:    {val_m['roc_auc']:.4f}")
print(f"Validation FPR:        {val_m['fpr']*100:.2f}%")


In [ ]:
# Plot Custom Experiment Confusion Matrix & Metric Breakdown
cm_matrix = np.array(val_m["confusion_matrix"])
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), dpi=100)

sns.heatmap(cm_matrix, annot=True, fmt="d", cmap="Blues", cbar=False, ax=axes[0],
            xticklabels=["Pred ADL", "Pred Fall"], yticklabels=["Actual ADL", "Actual Fall"])
axes[0].set_title(f"Validation Confusion Matrix ('{REPRESENTATION}')", fontweight="bold")

metrics = ["Accuracy", "Recall", "Precision", "Macro F1", "ROC-AUC"]
vals = [val_m["accuracy"], val_m["recall"], val_m["precision"], val_m["macro_f1"], val_m["roc_auc"]]
colors = ["#3498db", "#2ecc71", "#9b59b6", "#e74c3c", "#f39c12"]

bars = axes[1].bar(metrics, [v*100 if i < 4 else v for i, v in enumerate(vals)], color=colors, edgecolor="black", width=0.55)
axes[1].set_ylim(0, 115)
axes[1].set_title(f"Validation Metrics (epochs={epochs}, lr={lr})", fontweight="bold")
for bar in bars:
    h = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2., h + 2, f"{h:.1f}", ha="center", va="bottom", fontweight="bold")

plt.tight_layout()
plt.show()
